In [ ]:
%%capture
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.stats import shapiro, probplot
import seaborn as sns
import scipy.stats as stats
from dj_notebook import activate
from django_pandas.io import read_frame
from pathlib import Path

env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)
report_folder = Path(documents_folder)


In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858
from edc_pdutils.dataframes import get_crf, get_subject_visit, get_appointments
from edc_appointment.constants import SKIPPED_APPT, NEW_APPT

df_main = get_df_main_1858(None)


In [ ]:
from edc_model_to_dataframe import read_frame_edc
from intecomm_subject.models import Vitals

df_bp = read_frame_edc(Vitals.objects.all())


In [ ]:
df = df_bp[["subject_identifier", "visit_datetime", "sys_blood_pressure_avg", "sys_blood_pressure_one", "sys_blood_pressure_two","dia_blood_pressure_avg", "dia_blood_pressure_one", "dia_blood_pressure_two"]].copy()

In [ ]:
def get_dia_avg(s):
    new_avg = np.nan
    if pd.notna(s["dia_blood_pressure_one"]) and pd.notna(s["dia_blood_pressure_two"]):
        new_avg =  (s["dia_blood_pressure_one"] + s["dia_blood_pressure_two"]) / 2
    elif pd.notna(s["dia_blood_pressure_one"]) and pd.isna(s["dia_blood_pressure_two"]):
        new_avg = s["dia_blood_pressure_one"]
    return new_avg

def get_sys_avg(s):
    new_avg = np.nan
    if pd.notna(s["sys_blood_pressure_one"]) and pd.notna(s["sys_blood_pressure_two"]):
        new_avg =  (s["sys_blood_pressure_one"] + s["sys_blood_pressure_two"]) / 2
    elif pd.notna(s["sys_blood_pressure_one"]) and pd.isna(s["sys_blood_pressure_two"]):
        new_avg = s["sys_blood_pressure_one"]
    return new_avg


def categorize_bp(systolic, diastolic):
    if systolic < 90 or diastolic < 60:
        return 'L'
    elif (90 <= systolic < 120) and (60 <= diastolic < 80):
        return ''
    elif (120 <= systolic < 140) or (80 <= diastolic < 90):
        return 'M'
    elif (140 <= systolic < 180) or (90 <= diastolic < 110):
        return 'H'
    elif pd.isna(systolic) or pd.isna(diastolic):
        return ''
    else:
        return 'S!'

df["sys"] = df.apply(get_sys_avg, axis=1)
df["dia"] = df.apply(get_dia_avg, axis=1)
df['bp_category'] = df.apply(lambda row: categorize_bp(row['sys'], row['dia']), axis=1)

In [ ]:
df.sort_values(by=["subject_identifier","visit_datetime"], ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)


df

In [ ]:
df[df.bp_category=="S!"]

In [ ]:
# get a unique list of subjects
subjects = df[df.bp_category=="S!"]["subject_identifier"].unique()


In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
with PdfPages(os.path.expanduser('~/bp_output.pdf')) as pdf:
    for index, subject in enumerate(subjects):
        data = df[df["subject_identifier"] == subject]
        plt.figure(figsize=(6, 3))

        plt.axhline(y=139, color='g', linestyle='--')
        plt.axhline(y=80, color='g', linestyle='--')

        plt.plot(data['visit_datetime'], data['sys'], label='Sys', marker='o')
        plt.plot(data['visit_datetime'], data['dia'], label='Dia', marker='o')
        plt.fill_between(data['visit_datetime'], data['sys'], data['dia'], color='gray', alpha=0.2)
        for i in range(len(data)):
            plt.annotate(data.iloc[i]['bp_category'],
                         (data.iloc[i]['visit_datetime'], data.iloc[i]['sys'] + 2),
                         textcoords="offset points", xytext=(0,10), ha='center')


        plt.ylim(40, max(data['sys']) + 25 if max(data['sys']) > 200 else 200)

        plt.xticks(rotation=45)

        plt.xlabel('Date')
        plt.ylabel('Blood Pressure')
        plt.title(f'{subject}: {len(data["sys"])} readings', pad=20)
        # plt.legend()
        plt.subplots_adjust(left=0.1, right=0.9, top=0.7, bottom=0.2)
        # plt.tight_layout()
        # plt.show()
        pdf.savefig()
        plt.close()
        if index > 30:
            break

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd

In [ ]:
df = pd.DataFrame({
    'A': [1, 2, np.nan, 4],
    'B': [np.nan, 2, 3, 4],
    'C': [1, 2, 3, np.nan]
})

# Initialize and fit the IterativeImputer
imputer = IterativeImputer(max_iter=10, random_state=0)
imputed_df = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_df, columns=df.columns)

In [ ]:
imputed_df